## Part 1: iBeacon Only

In [ ]:
DATA_DIR = "./indoor_data"

In [ ]:
import subprocess, sys
subprocess.run([sys.executable, "-m", "pip", "install", "numpy", "scipy", "matplotlib", "-q"], check=True)
print("done")

In [ ]:
import glob, os
import numpy as np
from collections import defaultdict, Counter

def parse_trace(filepath):
    # reads a trace file, pulls out waypoints and beacon readings
    waypoints, beacons = [], []
    with open(filepath, "r", errors="ignore") as f:
        for line in f:
            parts = line.strip().split("\t")
            if len(parts) < 3 or line.startswith("#"):
                continue
            ts, kind = int(parts[0]), parts[1]
            if kind == "TYPE_WAYPOINT" and len(parts) >= 4:
                waypoints.append((ts, float(parts[2]), float(parts[3])))
            elif kind == "TYPE_BEACON" and len(parts) >= 8:
                bid = f"{parts[2]}_{parts[3]}_{parts[4]}"
                beacons.append((ts, bid, float(parts[6])))
    return waypoints, beacons

def match_to_position(waypoints, beacons):
    # for each beacon reading, find the closest waypoint by timestamp
    if not waypoints or not beacons:
        return []
    wp_ts = np.array([w[0] for w in waypoints])
    wp_xy = np.array([(w[1], w[2]) for w in waypoints])
    return [(*wp_xy[np.argmin(np.abs(wp_ts - ts))], rssi, bid)
            for ts, bid, rssi in beacons]

In [ ]:
SITE = "site1"
FLOORS = ["B1", "F1", "F2", "F3", "F4"]
floor_map = {"B1": 0, "F1": 1, "F2": 2, "F3": 3, "F4": 4}
N_BEACONS = 50
MISSING_RSS = -110

# collect readings from all floors to find top beacons across the building
all_readings_combined = []
for FLOOR in FLOORS:
    floor_path = os.path.join(DATA_DIR, "data", SITE, FLOOR, "path_data_files")
    for fp in glob.glob(os.path.join(floor_path, "*.txt")):
        wps, bkns = parse_trace(fp)
        all_readings_combined.extend(match_to_position(wps, bkns))

top_beacons = [bid for bid, _ in Counter(r[3] for r in all_readings_combined).most_common(N_BEACONS)]
beacon_idx  = {bid: i for i, bid in enumerate(top_beacons)}
print(f"unique beacons found: {len(top_beacons)}")

# build one feature vector per unique (x, y) position per floor
X_list, y_list = [], []
for FLOOR in FLOORS:
    floor_path = os.path.join(DATA_DIR, "data", SITE, FLOOR, "path_data_files")
    floor_readings = []
    for fp in glob.glob(os.path.join(floor_path, "*.txt")):
        wps, bkns = parse_trace(fp)
        floor_readings.extend(match_to_position(wps, bkns))

    pos_rss = defaultdict(dict)
    for (x, y, rssi, bid) in floor_readings:
        key = (round(x, 1), round(y, 1))
        if bid in beacon_idx:
            pos_rss[key][bid] = rssi

    for key, rss in pos_rss.items():
        row = [rss.get(bid, MISSING_RSS) for bid in top_beacons]
        row.append(floor_map[FLOOR])  # add floor as a feature
        X_list.append(row)
        y_list.append(list(key))

X = np.array(X_list, dtype=np.float32)
y = np.array(y_list, dtype=np.float32)
print(f"dataset: {X.shape[0]} samples, {X.shape[1]} features (last col = floor)")

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s  = scaler.transform(X_test)

print(f"train: {X_train_s.shape[0]}  test: {X_test_s.shape[0]}")

In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader

X_tr = torch.tensor(X_train_s)
y_tr = torch.tensor(y_train)
X_te = torch.tensor(X_test_s)
y_te = torch.tensor(y_test)

train_loader = DataLoader(TensorDataset(X_tr, y_tr), batch_size=32, shuffle=True)

class LocationMLP(nn.Module):
    def __init__(self, in_dim):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_dim, 128), nn.ReLU(), nn.Dropout(0.2),
            nn.Linear(128, 64),    nn.ReLU(), nn.Dropout(0.2),
            nn.Linear(64, 2)
        )
    def forward(self, x):
        return self.net(x)

model   = LocationMLP(X.shape[1])
opt     = torch.optim.Adam(model.parameters(), lr=1e-3)
loss_fn = nn.MSELoss()

EPOCHS = 100
for epoch in range(EPOCHS):
    model.train()
    for xb, yb in train_loader:
        pred = model(xb)
        loss = loss_fn(pred, yb)
        opt.zero_grad(); loss.backward(); opt.step()
    if (epoch + 1) % 20 == 0:
        model.eval()
        with torch.no_grad():
            test_loss = loss_fn(model(X_te), y_te)
        print(f"Epoch {epoch+1}/{EPOCHS}  train={loss.item():.2f}  test={test_loss.item():.2f}")

In [ ]:
import matplotlib.pyplot as plt

model.eval()
with torch.no_grad():
    preds = model(X_te).numpy()

errors = np.sqrt(((preds - y_test) ** 2).sum(axis=1))
print(f"Mean error  : {errors.mean():.2f} m")
print(f"Median error: {np.median(errors):.2f} m")
print(f"< 5m : {(errors < 5).mean()*100:.1f}%")
print(f"< 10m: {(errors < 10).mean()*100:.1f}%")

fig, ax = plt.subplots(figsize=(10, 8))
ax.scatter(y_test[:,0], y_test[:,1], s=20, alpha=0.5, label="ground truth", color="blue")
ax.scatter(preds[:,0],  preds[:,1],  s=20, alpha=0.5, label="predicted",    color="red")
ax.set_xlabel("X (m)"); ax.set_ylabel("Y (m)")
ax.set_title("iBeacon Only — Predicted vs Ground Truth")
ax.legend(); ax.set_aspect("equal")
plt.tight_layout()
plt.savefig("ibeacon_prediction.png", dpi=150)
plt.show()

## Part 2: Multimodal (iBeacon + WiFi + Geomagnetic)

In [ ]:
def parse_trace_multimodal(filepath):
    # same as before but also pulls wifi and magnetic field readings
    waypoints, beacons, wifi, magnetic = [], [], [], []
    with open(filepath, "r", errors="ignore") as f:
        for line in f:
            parts = line.strip().split("\t")
            if len(parts) < 3 or line.startswith("#"):
                continue
            ts, kind = int(parts[0]), parts[1]
            if kind == "TYPE_WAYPOINT" and len(parts) >= 4:
                waypoints.append((ts, float(parts[2]), float(parts[3])))
            elif kind == "TYPE_BEACON" and len(parts) >= 7:
                beacons.append((ts, f"{parts[2]}_{parts[3]}_{parts[4]}", float(parts[6])))
            elif kind == "TYPE_WIFI" and len(parts) >= 5:
                wifi.append((ts, parts[2], float(parts[4])))
            elif kind == "TYPE_MAGNETIC_FIELD" and len(parts) >= 5:
                magnetic.append((ts, float(parts[2]), float(parts[3]), float(parts[4])))
    return waypoints, beacons, wifi, magnetic

In [ ]:
SITE   = "site1"
FLOORS = ["B1", "F1", "F2", "F3", "F4"]
floor_map = {"B1": 0, "F1": 1, "F2": 2, "F3": 3, "F4": 4}
N_BEACON = 30
N_WIFI   = 30
MISSING  = -110

# pass 1: find top beacons and wifi APs across all floors
all_b, all_w = [], []
for FLOOR in FLOORS:
    floor_path = os.path.join(DATA_DIR, "data", SITE, FLOOR, "path_data_files")
    for fp in glob.glob(os.path.join(floor_path, "*.txt")):
        wps, bkns, wfis, mags = parse_trace_multimodal(fp)
        if not wps:
            continue
        wp_ts = np.array([w[0] for w in wps])
        wp_xy = np.array([(w[1], w[2]) for w in wps])
        for ts, bid, rssi in bkns:
            all_b.append((*wp_xy[np.argmin(np.abs(wp_ts - ts))], ts, bid, rssi))
        for ts, bssid, rssi in wfis:
            all_w.append((*wp_xy[np.argmin(np.abs(wp_ts - ts))], ts, bssid, rssi))

top_beacons = [b for b, _ in Counter(r[3] for r in all_b).most_common(N_BEACON)]
top_wifi    = [w for w, _ in Counter(r[3] for r in all_w).most_common(N_WIFI)]
print(f"top beacons: {len(top_beacons)}, top wifi APs: {len(top_wifi)}")

# pass 2: build feature vectors per position per floor
X_list, y_list = [], []
for FLOOR in FLOORS:
    floor_path = os.path.join(DATA_DIR, "data", SITE, FLOOR, "path_data_files")
    b_reads, w_reads, m_reads = [], [], []

    for fp in glob.glob(os.path.join(floor_path, "*.txt")):
        wps, bkns, wfis, mags = parse_trace_multimodal(fp)
        if not wps:
            continue
        wp_ts = np.array([w[0] for w in wps])
        wp_xy = np.array([(w[1], w[2]) for w in wps])
        for ts, bid, rssi in bkns:
            xy = wp_xy[np.argmin(np.abs(wp_ts - ts))]
            b_reads.append((*xy, ts, bid, rssi))
        for ts, bssid, rssi in wfis:
            xy = wp_xy[np.argmin(np.abs(wp_ts - ts))]
            w_reads.append((*xy, ts, bssid, rssi))
        for ts, mx, my, mz in mags:
            xy = wp_xy[np.argmin(np.abs(wp_ts - ts))]
            m_reads.append((*xy, ts, mx, my, mz))

    pos_data = defaultdict(lambda: {"b": defaultdict(list), "w": defaultdict(list), "m": []})
    for x, y, ts, bid, rssi in b_reads:
        key = (round(x, 1), round(y, 1))
        if bid in top_beacons:
            pos_data[key]["b"][bid].append(rssi)
    for x, y, ts, bssid, rssi in w_reads:
        key = (round(x, 1), round(y, 1))
        if bssid in top_wifi:
            pos_data[key]["w"][bssid].append(rssi)
    for x, y, ts, mx, my, mz in m_reads:
        pos_data[(round(x,1), round(y,1))]["m"].append((mx, my, mz))

    for (x, y), d in pos_data.items():
        b_feats = [np.mean(d["b"].get(b, [MISSING])) for b in top_beacons]
        w_feats = [np.mean(d["w"].get(w, [MISSING])) for w in top_wifi]
        m_feats = np.array(d["m"]).mean(axis=0).tolist() if d["m"] else [0.0, 0.0, 0.0]
        X_list.append(b_feats + w_feats + m_feats + [floor_map[FLOOR]])
        y_list.append([x, y])

X = np.array(X_list, dtype=np.float32)
y = np.array(y_list, dtype=np.float32)
print(f"dataset: {X.shape[0]} samples, {X.shape[1]} features")
print(f"  = {len(top_beacons)} beacon + {len(top_wifi)} wifi + 3 geomag + 1 floor")

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s  = scaler.transform(X_test)

print(f"train: {X_train_s.shape[0]}  test: {X_test_s.shape[0]}")

In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader

X_tr = torch.tensor(X_train_s)
y_tr = torch.tensor(y_train)
X_te = torch.tensor(X_test_s)
y_te = torch.tensor(y_test)

train_loader = DataLoader(TensorDataset(X_tr, y_tr), batch_size=32, shuffle=True)

class LocationMLP(nn.Module):
    def __init__(self, in_dim):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_dim, 128), nn.ReLU(), nn.Dropout(0.2),
            nn.Linear(128, 64),    nn.ReLU(), nn.Dropout(0.2),
            nn.Linear(64, 2)
        )
    def forward(self, x):
        return self.net(x)

model   = LocationMLP(X.shape[1])
opt     = torch.optim.Adam(model.parameters(), lr=1e-3)
loss_fn = nn.MSELoss()

EPOCHS = 300
for epoch in range(EPOCHS):
    model.train()
    for xb, yb in train_loader:
        pred = model(xb)
        loss = loss_fn(pred, yb)
        opt.zero_grad(); loss.backward(); opt.step()
    if (epoch + 1) % 20 == 0:
        model.eval()
        with torch.no_grad():
            test_loss = loss_fn(model(X_te), y_te)
        print(f"Epoch {epoch+1}/{EPOCHS}  train={loss.item():.2f}  test={test_loss.item():.2f}")

In [ ]:
import matplotlib.pyplot as plt

model.eval()
with torch.no_grad():
    preds = model(X_te).numpy()

errors = np.sqrt(((preds - y_test) ** 2).sum(axis=1))
print(f"Mean error  : {errors.mean():.2f} m")
print(f"Median error: {np.median(errors):.2f} m")
print(f"< 5m : {(errors < 5).mean()*100:.1f}%")
print(f"< 10m: {(errors < 10).mean()*100:.1f}%")

fig, ax = plt.subplots(figsize=(10, 8))
ax.scatter(y_test[:,0], y_test[:,1], s=20, alpha=0.5, label="ground truth", color="blue")
ax.scatter(preds[:,0],  preds[:,1],  s=20, alpha=0.5, label="predicted",    color="red")
ax.set_xlabel("X (m)"); ax.set_ylabel("Y (m)")
ax.set_title("Multimodal — Predicted vs Ground Truth")
ax.legend(); ax.set_aspect("equal")
plt.tight_layout()
plt.savefig("multimodal_prediction.png", dpi=150)
plt.show()